In [ ]:
# setup Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import multiprocessing

# Tự động đếm số lõi CPU thực tế trên máy bạn
cores = multiprocessing.cpu_count()
# Cấu hình số partition thường gấp 2-3 lần số lõi CPU để tối ưu hóa
safe_cores = max(4, int(cores * 0.6))
num_partitions = safe_cores * 3
# OPTIMIZED Spark Configuration
spark = SparkSession.builder \
    .appName("Amazon Review Local Processing") \
    .master(f"local[{safe_cores}]") \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "70MB") \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "1g") \
    .getOrCreate()
print(f"✅ Spark {spark.version} initialized for LOCAL MODE")
print(f"🖥️  CPU Cores utilized: {cores}")
print(f"📊 Partitions configured: {num_partitions}")
print(f"💾 Driver Memory (Max RAM): 10GB")

✅ Spark 3.5.1 initialized for LOCAL MODE
🖥️  CPU Cores utilized: 16
📊 Partitions configured: 27
💾 Driver Memory (Max RAM): 8GB


In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import DataFrame
import pandas as pd
import numpy as np
from pyspark.ml.feature import StringIndexer

In [3]:
from pathlib import Path

ROOT_DIR      = Path().resolve().parent
DATA_DIR      = ROOT_DIR / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Đảm bảo thư mục processed tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 📥 Đường dẫn đầu vào (Ưu tiên dùng Parquet đã convert để nhanh hơn)
REVIEW_RAW_PARQUET = str(RAW_DIR / "Clothing_Shoes_and_Jewelry.parquet")
META_RAW_PARQUET   = str(RAW_DIR / "meta_Clothing_Shoes_and_Jewelry.parquet")

# 📤 Đường dẫn đầu ra cho các bước xử lý tiếp theo
REVIEW_CLEAN_PARQUET = str(PROCESSED_DIR / "review_clean_spark.parquet")
META_CLEAN_PARQUET   = str(PROCESSED_DIR / "meta_clean_spark.parquet")
FINAL_REVIEWS_PARQUET = str(PROCESSED_DIR / "final_kcore_reviews.parquet")
FINAL_META_PARQUET = str(PROCESSED_DIR / "final_kcore_metadata.parquet")

# Các tham số cấu hình khác
K_CORE = 5
OUTLIER_ZSCORE_THRESHOLD = 3.0
MIN_TEXT_LENGTH = 10
MAX_TEXT_LENGTH = 5000
NUM_PARTITIONS = 200
print(f"📂 Project Root: {ROOT_DIR}")
print(f"✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.")

📂 Project Root: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis
✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.


In [4]:
import math
from pyspark.sql import functions as F

print("\n" + "="*60)
print("🚀 BẮT ĐẦU QUY TRÌNH XUẤT DỮ LIỆU CHO KAGGLE/COLAB")
print("="*60)
df_final_meta = spark.read.parquet(FINAL_META_PARQUET)
df_final_reviews = spark.read.parquet(FINAL_REVIEWS_PARQUET)
# -----------------------------------------------------------
# 1. TỈA CỘT METADATA
# -----------------------------------------------------------
cols_meta = [
    'parent_asin',    # Khóa (Bắt buộc để Join)
    'price',          # Biến số
    'average_rating', # Biến số
    'rating_number',  # Biến số
    'main_category',  # Biến phân loại
    'price_category', # Biến phân loại
    'store'           # Biến phân loại
]

# Tỉa cột và xóa duplicate
df_meta_final = df_final_meta.select(
    [c for c in cols_meta if c in df_final_meta.columns]
).dropDuplicates(['parent_asin'])

print(f"✅ Metadata columns kept: {df_meta_final.columns}")

# ===========================================================
# 2. MERGE DỮ LIỆU VÀ ĐÓNG BĂNG (CACHE)
# ===========================================================
print("\n🔗 Đang merge reviews + metadata và tải vào RAM...")

df_master = df_final_reviews.join(
    df_meta_final, 
    on='parent_asin', 
    how='inner'
)

# BƯỚC CỨU MẠNG: Đóng băng dữ liệu sau khi xáo trộn để giữ nguyên thứ tự
df_master = df_master.cache()

# Kích hoạt tính toán và lưu vào RAM bằng 1 lệnh count duy nhất
total_rows = df_master.count() 
print(f"   Sau merge: {total_rows:,} reviews")



🚀 BẮT ĐẦU QUY TRÌNH XUẤT DỮ LIỆU CHO KAGGLE/COLAB
✅ Metadata columns kept: ['parent_asin', 'price', 'average_rating', 'rating_number', 'main_category', 'price_category', 'store']

🔗 Đang merge reviews + metadata và tải vào RAM...
   Sau merge: 23,202,904 reviews


In [7]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# ===========================================================
# 3. MÃ HÓA ID BẰNG NATIVE SPARK SQL (100% TRONG JVM, KHÔNG CRASH PYTHON)
# ===========================================================
print("\n🔄 Đang mã hóa user_id và parent_asin thành Integer (Native SQL Mode)...")

def create_distributed_indexer(df, col_name, new_col_name):
    print(f"   Đang tạo từ điển liên tục cho {col_name}...")
    
    # 1. Lấy danh sách ID duy nhất (Chạy phân tán)
    unique_df = df.select(col_name).distinct()
    
    # 2. Sử dụng Window để đánh số. 
    # Việc distinct ở trên đã làm giảm lượng data từ 23 triệu xuống còn rất nhỏ
    # Nên dùng Window orderBy ở đây cực kỳ an toàn và siêu tốc.
    w = Window.orderBy(F.monotonically_increasing_id())
    
    # 3. Dùng row_number() đếm từ 1, trừ đi 1 để index bắt đầu từ 0
    mapping_df = unique_df.withColumn(new_col_name, (F.row_number().over(w) - 1).cast("integer"))
    
    return mapping_df

# Tạo bảng từ điển User và Item
user_mapping = create_distributed_indexer(df_master, "user_id", "user_id_int")
item_mapping = create_distributed_indexer(df_master, "parent_asin", "item_id_int")

# Dùng hàm JOIN để ghép ID mới vào bảng Master
df_master = df_master.join(user_mapping, on="user_id", how="left")
df_master = df_master.join(item_mapping, on="parent_asin", how="left")

# Đóng băng DataFrame để chuẩn bị Random Split an toàn
df_master = df_master.cache()
print(f"   ✅ Đã mã hóa xong! Kích thước Master: {df_master.count():,} dòng.")


🔄 Đang mã hóa user_id và parent_asin thành Integer (Native SQL Mode)...
   Đang tạo từ điển liên tục cho user_id...
   Đang tạo từ điển liên tục cho parent_asin...
   ✅ Đã mã hóa xong! Kích thước Master: 23,202,904 dòng.


In [8]:
# ===========================================================
# 4. CHIA TẬP DỮ LIỆU (RANDOM SPLIT)
# ===========================================================
print("\n✂️ Đang chia tập dữ liệu (Train 70%, Val 15%, Test 15%)...")

# Chia split lúc này cực kỳ an toàn, không sợ trùng lặp hay mất dữ liệu
splits = df_master.randomSplit([0.7, 0.15, 0.15], seed=42)
train_df = splits[0]
val_df = splits[1]
test_df = splits[2]

print("   ✅ Đã chia xong!")

# ===========================================================
# 5. XUẤT FILE PARQUET (KHÔNG LẶP LẠI COUNT, DÙNG COALESCE)
# ===========================================================
TRAIN_PATH = str(PROCESSED_DIR / "train_data.parquet")
VAL_PATH   = str(PROCESSED_DIR / "val_data.parquet")
TEST_PATH  = str(PROCESSED_DIR / "test_data.parquet")

def export_parquet(df, path, name, num_part):
    print(f"\n💾 Đang xuất {name}...")
    
    # Dùng coalesce thay vì repartition để tránh xáo trộn (Shuffle) lại từ đầu
    df.coalesce(num_part).write.mode('overwrite').parquet(path)
    
    print(f"   ✅ Đã lưu thành công tại: {path}")

# Truyền trực tiếp số lượng phân vùng đã tính nhẩm để tiết kiệm RAM
export_parquet(train_df, TRAIN_PATH, "TẬP TRAIN (70%)", num_part=10)
export_parquet(val_df, VAL_PATH, "TẬP VALIDATION (15%)", num_part=2)
export_parquet(test_df, TEST_PATH, "TẬP TEST (15%)", num_part=2)

# ===========================================================
# 6. DỌN DẸP
# ===========================================================
df_master.unpersist()
print("\n✨ HOÀN THÀNH XUẤT DỮ LIỆU! Mọi thứ đã sẵn sàng đưa lên Kaggle.")


✂️ Đang chia tập dữ liệu (Train 70%, Val 15%, Test 15%)...
   ✅ Đã chia xong!

💾 Đang xuất TẬP TRAIN (70%)...
   ✅ Đã lưu thành công tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\train_data.parquet

💾 Đang xuất TẬP VALIDATION (15%)...
   ✅ Đã lưu thành công tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\val_data.parquet

💾 Đang xuất TẬP TEST (15%)...
   ✅ Đã lưu thành công tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\test_data.parquet

✨ HOÀN THÀNH XUẤT DỮ LIỆU! Mọi thứ đã sẵn sàng đưa lên Kaggle.
